In [58]:
import importlib
import metrics
importlib.reload(metrics)

Imports finis


<module 'metrics' from 'c:\\Users\\leoet\\Documents\\Scolarité\\CentraleSupélec\\3A\\Challenge\\llm-creativity-eval\\metrics.py'>

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from collections import Counter
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, CamembertTokenizer, CamembertForMaskedLM
import torch
import torch.nn.functional as F
from torch.nn.functional import cosine_similarity
from rouge_score import rouge_scorer
from nltk.tokenize import sent_tokenize
import math

In [2]:
table = pq.read_table('../compria-reactions/reactions.parquet')
table = table.cast(
    pa.schema([(f.name, pa.string() if pa.types.is_large_string(f.type) else f.type) 
               for f in table.schema])
)
df_reactions = table.to_pandas()

In [4]:
print(np.shape(df_reactions))
df_sampled = df_reactions.sample(n=1000)
print(np.shape(df_sampled))

(94294, 34)
(1000, 34)


In [5]:
model_sentence = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
tokenizer_sentence = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
ref_model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(ref_model_name)
model = AutoModelForCausalLM.from_pretrained(ref_model_name)
nlp = spacy.load("fr_core_news_sm")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [8]:
df_sampled["index_moving_average_ttr"] = df_sampled['response_content'].apply(lambda x:metrics.index_moving_average_ttr(x,nlp))

In [80]:
df_sampled.columns

Index(['id', 'timestamp', 'model_a_name', 'model_b_name', 'refers_to_model',
       'msg_index', 'opening_msg', 'conversation_a', 'conversation_b',
       'model_pos', 'conv_turns', 'conversation_pair_id', 'conv_a_id',
       'conv_b_id', 'refers_to_conv_id', 'session_hash', 'visitor_id',
       'response_content', 'question_content', 'liked', 'disliked', 'comment',
       'useful', 'creative', 'complete', 'clear_formatting', 'incorrect',
       'superficial', 'instructions_not_followed', 'model_pair_name',
       'msg_rank', 'question_id', 'system_prompt', '__index_level_0__',
       'index_moving_average_ttr', 'index_coherence_locale',
       'index_surprise_mean', 'index_surprise_mean_log',
       'index_surprise_mean_exp', 'index_surprise_mean_norm',
       'index_creativity'],
      dtype='str')

In [33]:
df_sampled["index_coherence_locale"] = df_sampled['response_content'].apply(lambda x:metrics.index_coherence_locale(x,tokenizer_sentence,model_sentence))

c:\Users\leoet\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\leoet\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [34]:
df_sampled.head()

,id,timestamp,model_a_name,model_b_name,refers_to_model,msg_index,opening_msg,conversation_a,conversation_b,model_pos,...,incorrect,superficial,instructions_not_followed,model_pair_name,msg_rank,question_id,system_prompt,__index_level_0__,index_moving_average_ttr,index_coherence_locale
61430,302316,2025-10-15 06:03:33.888662,Qwen3-Coder-480B-A35B-Instruct,aya-expanse-32b,Qwen3-Coder-480B-A35B-Instruct,1,A partir du tableau des recettes concernant le...,[{'content': 'A partir du tableau des recettes...,[{'content': 'A partir du tableau des recettes...,a,...,False,False,False,"[Qwen3-Coder-480B-A35B-Instruct, aya-expanse-32b]",0,f97c95dc36244c9ead59dcef5f3d6c2b-f9e6ddac73004...,NaN,61436,0.509842,0.411605
4643,182006,2025-03-29 14:21:05.737361,c4ai-command-r-08-2024,mistral-nemo-2407,mistral-nemo-2407,1,où louer un vélo électrique ?,"[{'content': 'où louer un vélo électrique ?', ...","[{'content': 'où louer un vélo électrique ?', ...",b,...,False,False,False,"[c4ai-command-r-08-2024, mistral-nemo-2407]",0,d25ad83e1eae43e68857b997e885a1ff-e3d00bb1030a4...,NaN,4643,0.602500,0.153099
88638,432229,2026-01-25 07:11:42.132142,Qwen3-Coder-480B-A35B-Instruct,gemma-3-12b,gemma-3-12b,1,Pourquoi lançaient-on de l'huile bouillante du...,[{'content': 'Pourquoi lançaient-on de l'huile...,[{'content': 'Pourquoi lançaient-on de l'huile...,b,...,False,False,False,"[Qwen3-Coder-480B-A35B-Instruct, gemma-3-12b]",0,39404e94be114f23a0f6d8967e638d16-225aa4f78cb64...,NaN,88647,0.659091,0.358009
86794,460297,2026-02-27 15:07:10.369862,gemma-3-27b,gpt-5-mini,gemma-3-27b,1,Interprète moi l'équation de regression suivan...,[{'content': 'Interprète moi l'équation de reg...,[{'content': 'Interprète moi l'équation de reg...,a,...,False,False,False,"[gemma-3-27b, gpt-5-mini]",0,60865f6217ec4d24b62c24a4d8bf8357-608cb16a63db4...,NaN,86803,0.594378,0.430359
83580,404828,2026-01-09 12:56:57.222827,kimi-k2-thinking,mistral-medium-2508,mistral-medium-2508,7,qui sont les producteurs mondiaux d'imprimante...,[{'content': 'qui sont les producteurs mondiau...,[{'content': 'qui sont les producteurs mondiau...,b,...,False,False,False,"[kimi-k2-thinking, mistral-medium-2508]",3,67168b4d3e744207b949666598f690b5-57de6584d7d24...,NaN,83587,0.477472,0.406172


In [59]:
df_sampled["index_surprise_mean"] = df_sampled['response_content'].apply(lambda x:metrics.index_surprise_mean(x,tokenizer,model))

done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


In [71]:
df_sampled["index_surprise_mean_norm"] = df_sampled["index_surprise_mean"].apply(lambda x : x)
df_sampled["index_surprise_mean_norm"] = (df_sampled["index_surprise_mean_norm"]-df_sampled["index_surprise_mean_norm"].min())/(df_sampled["index_surprise_mean_norm"].max()-df_sampled["index_surprise_mean_norm"].min())

In [72]:
df_sampled.head()

,id,timestamp,model_a_name,model_b_name,refers_to_model,msg_index,opening_msg,conversation_a,conversation_b,model_pos,...,msg_rank,question_id,system_prompt,__index_level_0__,index_moving_average_ttr,index_coherence_locale,index_surprise_mean,index_surprise_mean_log,index_surprise_mean_exp,index_surprise_mean_norm
61430,302316,2025-10-15 06:03:33.888662,Qwen3-Coder-480B-A35B-Instruct,aya-expanse-32b,Qwen3-Coder-480B-A35B-Instruct,1,A partir du tableau des recettes concernant le...,[{'content': 'A partir du tableau des recettes...,[{'content': 'A partir du tableau des recettes...,a,...,0,f97c95dc36244c9ead59dcef5f3d6c2b-f9e6ddac73004...,NaN,61436,0.509842,0.411605,3.978144,NaN,0.018720,0.336896
4643,182006,2025-03-29 14:21:05.737361,c4ai-command-r-08-2024,mistral-nemo-2407,mistral-nemo-2407,1,où louer un vélo électrique ?,"[{'content': 'où louer un vélo électrique ?', ...","[{'content': 'où louer un vélo électrique ?', ...",b,...,0,d25ad83e1eae43e68857b997e885a1ff-e3d00bb1030a4...,NaN,4643,0.602500,0.153099,4.067865,NaN,0.017114,0.347009
88638,432229,2026-01-25 07:11:42.132142,Qwen3-Coder-480B-A35B-Instruct,gemma-3-12b,gemma-3-12b,1,Pourquoi lançaient-on de l'huile bouillante du...,[{'content': 'Pourquoi lançaient-on de l'huile...,[{'content': 'Pourquoi lançaient-on de l'huile...,b,...,0,39404e94be114f23a0f6d8967e638d16-225aa4f78cb64...,NaN,88647,0.659091,0.358009,5.299367,NaN,0.004995,0.485823
86794,460297,2026-02-27 15:07:10.369862,gemma-3-27b,gpt-5-mini,gemma-3-27b,1,Interprète moi l'équation de regression suivan...,[{'content': 'Interprète moi l'équation de reg...,[{'content': 'Interprète moi l'équation de reg...,a,...,0,60865f6217ec4d24b62c24a4d8bf8357-608cb16a63db4...,NaN,86803,0.594378,0.430359,4.899849,NaN,0.007448,0.440790
83580,404828,2026-01-09 12:56:57.222827,kimi-k2-thinking,mistral-medium-2508,mistral-medium-2508,7,qui sont les producteurs mondiaux d'imprimante...,[{'content': 'qui sont les producteurs mondiau...,[{'content': 'qui sont les producteurs mondiau...,b,...,3,67168b4d3e744207b949666598f690b5-57de6584d7d24...,NaN,83587,0.477472,0.406172,2.351825,NaN,0.095195,0.153577


In [73]:
df_sampled["index_creativity"] = 0.15*df_sampled["index_moving_average_ttr"]+0.35*df_sampled["index_coherence_locale"]+0.5*df_sampled["index_surprise_mean_norm"]

In [84]:
df_sampled["creative_num"] = df_sampled["creative"].astype(float)
df_sampled["creative_num"]

61430    0.0
4643     0.0
88638    0.0
86794    0.0
83580    0.0
        ... 
93689    0.0
76147    0.0
74289    0.0
4942     0.0
40974    0.0
Name: creative_num, Length: 1000, dtype: float64

In [78]:
df_sampled["index_creativity"]

61430    0.388986
4643     0.317464
88638    0.467079
86794    0.460177
83580    0.290570
           ...   
93689    0.609792
76147    0.502877
74289    0.545131
4942     0.509072
40974    0.475796
Name: index_creativity, Length: 1000, dtype: float64

In [86]:
df_sampled["index_creativity"].corr(df_sampled["creative_num"])

np.float64(0.017006113361869725)